In [10]:
if 'spark' in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/12 14:36:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "kafka"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-07-13-Jul"
TABLE_NAME = "videos"

execution_date = "2026-07-13-Jul"
full_file_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}"

#Pv = "s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_data=2026-07-13-Jul/videos.parquet"

In [3]:
df_events = spark.read.parquet(full_file_path).cache()
df_events.createOrReplaceTempView("events")

26/08/12 14:37:08 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [6]:
df_silver = spark.sql("""
SELECT
    event_id,
    user_id,
    video_id,
    interaction_type,
    device_type,

    CAST(watch_time_sec AS INT) AS watch_time_sec,

    TO_TIMESTAMP(event_timestamp) AS event_timestamp,

    TO_DATE(TO_TIMESTAMP(event_timestamp)) AS event_date,

    HOUR(TO_TIMESTAMP(event_timestamp)) AS event_hour,

    (
        interaction_type IN ('play', 'complete')
        AND watch_time_sec IS NULL
    ) AS dq_missing_watch_time,

    (
        interaction_type IN ('like', 'pause')
        AND watch_time_sec IS NOT NULL
    ) AS dq_unexpected_watch_time,

    (
        watch_time_sec IS NOT NULL
        AND (
            CAST(watch_time_sec AS INT) <= 0
            OR CAST(watch_time_sec AS INT) > 120
        )
    ) AS dq_invalid_watch_time,

    (
        interaction_type IS NULL
        OR interaction_type NOT IN ('like', 'play', 'complete', 'pause')
    ) AS dq_invalid_interaction,

    (
        device_type IS NULL
        OR device_type NOT IN ('mobile', 'tv', 'web')
    ) AS dq_invalid_device,

    (
        TO_TIMESTAMP(event_timestamp) IS NULL
    ) AS dq_invalid_timestamp,

    CURRENT_DATE() AS ingestion_date,

    CONCAT(
        DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMdd_HHmmss'),
        '_kafka'
    ) AS batch_id

FROM events
""")

SyntaxError: unmatched ')' (1735057309.py, line 57)

In [7]:
df_silver.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- interaction_type: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- watch_time_sec: integer (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- dq_missing_watch_time: boolean (nullable = true)
 |-- dq_unexpected_watch_time: boolean (nullable = true)
 |-- dq_invalid_watch_time: boolean (nullable = true)
 |-- dq_invalid_interaction: boolean (nullable = true)
 |-- dq_invalid_device: boolean (nullable = true)
 |-- dq_invalid_timestamp: boolean (nullable = false)
 |-- ingestion_date: date (nullable = false)
 |-- batch_id: string (nullable = false)



In [8]:
df_silver.show(3,truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------
 event_id                 | 8c6973d5-0dae-42ad-906a-a6c57685d7d8 
 user_id                  | 3dc19c4b-098c-4791-b993-93a22c544d97 
 video_id                 | 33ea938f-667b-439a-8e70-99cf6d78007f 
 interaction_type         | pause                                
 device_type              | mobile                               
 watch_time_sec           | NULL                                 
 event_timestamp          | 2026-07-13 19:23:14.473538           
 event_date               | 2026-07-13                           
 event_hour               | 19                                   
 dq_missing_watch_time    | false                                
 dq_unexpected_watch_time | false                                
 dq_invalid_watch_time    | false                                
 dq_invalid_interaction   | false                                
 dq_invalid_device        | false                                
 dq_invali